In [ ]:
import numpy as np
import re
from collections import Counter, defaultdict


In [ ]:

STOPWORDS = {
    "el", "la", "los", "las", "un", "una", "unos", "unas",
    "de", "del", "en", "a", "y", "que", "se", "es", "con",
    "por", "para", "como", "su", "sus", "al", "lo", "mas",
    "sin", "sobre", "entre", "también", "muy",
}

def preprocesar(texto):
    """Minúsculas → quitar signos → tokenizar → quitar stopwords"""
    texto  = texto.lower()
    texto  = re.sub(r'[^a-záéíóúüñ\s]', '', texto)
    tokens = texto.split()
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]



In [ ]:

# ══════════════════════════════════════════════════════════════
#  COMPONENTE 1: Word2Vec Skip-Gram
# ══════════════════════════════════════════════════════════════
class Word2VecSkipGram:
    def __init__(self, dim=20, ventana=2, lr=0.05):
        self.dim     = dim
        self.ventana = ventana
        self.lr      = lr

    def construir_vocabulario(self, corpus_tokens):
        conteo     = Counter(t for sent in corpus_tokens for t in sent)
        self.vocab = sorted(conteo.keys())
        self.w2i   = {w: i for i, w in enumerate(self.vocab)}
        self.i2w   = {i: w for w, i in self.w2i.items()}
        self.V     = len(self.vocab)

    def inicializar_pesos(self):
        self.W  = np.random.uniform(-0.5, 0.5, (self.V, self.dim)) / self.dim
        self.W2 = np.random.uniform(-0.5, 0.5, (self.dim, self.V)) / self.dim

    def softmax(self, x):
        e = np.exp(x - np.max(x))
        return e / e.sum()

    def forward(self, centro_idx):
        h = self.W[centro_idx]
        z = self.W2.T @ h
        y = self.softmax(z)
        return h, z, y

    def backward(self, centro_idx, ctx_idx, h, y):
        e  = y.copy()
        e[ctx_idx] -= 1
        dW2 = np.outer(h, e)
        dh  = self.W2 @ e
        self.W2            -= self.lr * dW2
        self.W[centro_idx] -= self.lr * dh

    def entrenar(self, corpus_tokens, epocas=100):
        self.construir_vocabulario(corpus_tokens)
        self.inicializar_pesos()
        pares = []
        for sent in corpus_tokens:
            idx = [self.w2i[w] for w in sent if w in self.w2i]
            for i, centro in enumerate(idx):
                inicio = max(0, i - self.ventana)
                fin    = min(len(idx), i + self.ventana + 1)
                for j in range(inicio, fin):
                    if j != i:
                        pares.append((centro, idx[j]))
        print(f"  Pares de entrenamiento: {len(pares)}")
        for ep in range(epocas):
            np.random.shuffle(pares)
            perdida = 0
            for centro_idx, ctx_idx in pares:
                h, z, y = self.forward(centro_idx)
                perdida -= np.log(y[ctx_idx] + 1e-9)
                self.backward(centro_idx, ctx_idx, h, y)
            if (ep + 1) % 50 == 0:
                print(f"  Época {ep+1}/{epocas}  |  Pérdida: {perdida:.4f}")

    def embedding(self, palabra):
        return self.W[self.w2i[palabra]]

    def similares(self, palabra, n=5):
        vec  = self.embedding(palabra)
        sims = []
        for w in self.vocab:
            if w == palabra:
                continue
            v2  = self.W[self.w2i[w]]
            cos = np.dot(vec, v2) / (np.linalg.norm(vec) * np.linalg.norm(v2) + 1e-9)
            sims.append((w, cos))
        return sorted(sims, key=lambda x: -x[1])[:n]



In [ ]:

# ══════════════════════════════════════════════════════════════
#  COMPONENTE 2: Naive Bayes para clasificación de texto
# ══════════════════════════════════════════════════════════════
class NaiveBayesSentimiento:
    def entrenar(self, documentos, etiquetas):
        self.clases  = list(set(etiquetas))
        self.log_prior   = {}
        self.log_likeli  = {}
        self.vocab       = set()

        # Agrupar tokens por clase
        tokens_clase = defaultdict(list)
        conteo_clase = Counter(etiquetas)
        total_docs   = len(etiquetas)

        for doc, etq in zip(documentos, etiquetas):
            tokens_clase[etq].extend(preprocesar(doc))

        for tokens in tokens_clase.values():
            self.vocab.update(tokens)

        V = len(self.vocab)

        for clase in self.clases:
            # Prior: P(clase)
            self.log_prior[clase] = np.log(conteo_clase[clase] / total_docs)
            # Verosimilitud con suavizado de Laplace
            conteo = Counter(tokens_clase[clase])
            total  = sum(conteo.values()) + V
            self.log_likeli[clase] = {
                w: np.log((conteo.get(w, 0) + 1) / total)
                for w in self.vocab
            }
            # Término para palabras desconocidas
            self.log_likeli[clase]['<UNK>'] = np.log(1 / total)

    def clasificar(self, texto):
        tokens = preprocesar(texto)
        scores = {}
        for clase in self.clases:
            score = self.log_prior[clase]
            for t in tokens:
                key    = t if t in self.log_likeli[clase] else '<UNK>'
                score += self.log_likeli[clase][key]
            scores[clase] = score
        mejor = max(scores, key=scores.get)
        return mejor, scores



In [ ]:

# ══════════════════════════════════════════════════════════════
#  COMPONENTE 3: Motor de búsqueda NLP
# ══════════════════════════════════════════════════════════════
class MotorBusquedaNLP:
    def __init__(self):
        self.w2v          = Word2VecSkipGram(dim=20, ventana=2, lr=0.05)
        self.clasificador = NaiveBayesSentimiento()
        self.corpus       = []
        self.corpus_tokens= []

    def indexar(self, documentos, etiquetas=None):
        """Preprocesa e indexa el corpus. Entrena embeddings y clasificador."""
        self.corpus        = documentos
        self.corpus_tokens = [preprocesar(d) for d in documentos]

        print("Entrenando embeddings...")
        self.w2v.entrenar(self.corpus_tokens, epocas=200)

        if etiquetas:
            print("Entrenando clasificador...")
            self.clasificador.entrenar(documentos, etiquetas)

        # Pre-calcular vector promedio por documento
        self.doc_vecs = []
        for tokens in self.corpus_tokens:
            vecs = [self.w2v.embedding(t) for t in tokens if t in self.w2v.w2i]
            if vecs:
                self.doc_vecs.append(np.mean(vecs, axis=0))
            else:
                self.doc_vecs.append(np.zeros(self.w2v.dim))

    def buscar(self, consulta, n=3):
        """Retorna los n documentos más similares a la consulta."""
        tokens_q = preprocesar(consulta)
        vecs_q   = [self.w2v.embedding(t) for t in tokens_q if t in self.w2v.w2i]
        if not vecs_q:
            return []
        vec_q = np.mean(vecs_q, axis=0)
        sims  = []
        for i, dv in enumerate(self.doc_vecs):
            cos = np.dot(vec_q, dv) / (
                np.linalg.norm(vec_q) * np.linalg.norm(dv) + 1e-9)
            sims.append((i, cos, self.corpus[i]))
        return sorted(sims, key=lambda x: -x[1])[:n]

    def analizar(self, texto):
        """Clasifica la categoría y sugiere documentos relacionados."""
        clase, scores = self.clasificador.clasificar(texto)
        similares     = self.buscar(texto)
        return {
            'sentimiento'          : clase,
            'scores'               : scores,
            'documentos_similares' : similares,
        }



In [ ]:

# ══════════════════════════════════════════════════════════════
#  EJECUCIÓN
# ══════════════════════════════════════════════════════════════
if __name__ == '__main__':

    noticias = [
        'python se consolida como el lenguaje mas usado en inteligencia artificial',
        'las redes neuronales aprenden representaciones del lenguaje natural',
        'el procesamiento de lenguaje natural avanza gracias a los transformers',
        'java sigue siendo popular en el desarrollo de aplicaciones empresariales',
        'rust ofrece seguridad de memoria sin recolector de basura',
        'los modelos de lenguaje grandes requieren enormes recursos computacionales',
        'el aprendizaje automatico transforma la industria del software',
        'python tiene librerias para machine learning como sklearn y pytorch',
    ]
    etiquetas = ['tech', 'ia', 'ia', 'tech', 'tech', 'ia', 'ia', 'tech']

    motor = MotorBusquedaNLP()
    motor.indexar(noticias, etiquetas)

    # ── Búsqueda semántica ──────────────────────────────────
    consulta = 'redes neuronales y aprendizaje profundo'
    print(f'\nBúsqueda: "{consulta}"')
    for idx, sim, doc in motor.buscar(consulta, n=3):
        print(f'  [{sim:.3f}] {doc[:65]}')

    # ── Análisis de una noticia nueva ───────────────────────
    nueva = 'los transformers revolucionaron el procesamiento de texto'
    resultado = motor.analizar(nueva)
    print(f'\nAnálisis de: "{nueva}"')
    print(f'  Categoría predicha : {resultado["sentimiento"]}')
    print(f'  Scores             : { {k: round(v,2) for k,v in resultado["scores"].items()} }')
    print('  Documentos similares:')
    for _, sim, doc in resultado['documentos_similares']:
        print(f'    [{sim:.3f}] {doc[:60]}')